# Kannada SSL — Model Evaluation

Evaluate trained models: wav2vec2 (speech) and BERT text classifier.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

from config import Config
from models.text_classifier import KannadaTextClassifier
from models.wav2vec2_model import KannadaWav2Vec2Model
from models.translator import full_pipeline

config = Config()
plt.style.use('dark_background')
print('Evaluation notebook ready!')

## 1. Training History Plots

In [ ]:
import os

def plot_training_history(history_path, title='Training History'):
    if not os.path.exists(history_path):
        print(f'No history found at {history_path}. Train the model first.')
        return
    
    with open(history_path) as f:
        history = json.load(f)
    
    epochs = [h['epoch'] for h in history]
    losses = [h['loss'] for h in history]
    
    fig, axes = plt.subplots(1, 2 if 'val_accuracy' in history[0] else 1, figsize=(14, 5))
    if not isinstance(axes, np.ndarray):
        axes = [axes]
    
    axes[0].plot(epochs, losses, color='#6366f1', linewidth=2.5, marker='o', markersize=4)
    axes[0].fill_between(epochs, losses, alpha=0.1, color='#6366f1')
    axes[0].set_title(f'{title} — Loss', fontsize=13)
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.2)
    
    if len(axes) > 1 and 'val_accuracy' in history[0]:
        accs = [h['val_accuracy'] for h in history]
        axes[1].plot(epochs, accs, color='#10b981', linewidth=2.5, marker='o', markersize=4)
        axes[1].fill_between(epochs, accs, alpha=0.1, color='#10b981')
        axes[1].set_title(f'{title} — Validation Accuracy', fontsize=13)
        axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
        axes[1].grid(True, alpha=0.2)
    
    plt.tight_layout()
    plt.show()

plot_training_history(
    os.path.join(config.WAV2VEC2_KANNADA_CHECKPOINT, 'training_history.json'),
    'wav2vec2'
)
plot_training_history(
    os.path.join(config.TEXT_CLASSIFIER_CHECKPOINT, 'training_history.json'),
    'Text Classifier'
)

## 2. Multilingual Pipeline Test

In [ ]:
test_inputs = [
    ('Hello, how are you?', 'en', 'hi'),
    ('नमस्कार, कन्नड बहुत सुंदर भाषा है।', 'hi', 'en'),
    ('Bonjour le monde', 'fr', 'kn'),
    ('ಕನ್ನಡ ನಮ್ಮ ತಾಯ್ನುಡಿ', 'kn', 'te'),
]

for text, src, tgt in test_inputs:
    result = full_pipeline(text, target_language=tgt)
    print(f'Input ({src}): {text}')
    print(f'Kannada: {result.get("kannada_text", "—")}')
    print(f'Output ({tgt}): {result.get("final_output", "—")}')
    print('—' * 60)

## 3. Text Classifier Evaluation

In [ ]:
classifier = KannadaTextClassifier(
    checkpoint_path=os.path.join(config.TEXT_CLASSIFIER_CHECKPOINT, 'best_model')
)

if classifier.load():
    test_texts = [
        'ಕ್ರಿಕೆಟ್ ತಂಡ ಇಂದು ಚಾಂಪಿಯನ್ ಆಯಿತು',  # Sports
        'ಸರ್ಕಾರ ಹೊಸ ನೀತಿ ಘೋಷಿಸಿತು',           # Politics
        'ಚಲನಚಿತ್ರ ಇಂದು ತೆರೆ ಕಂಡಿತು',             # Entertainment
    ]
    
    for text in test_texts:
        result = classifier.classify(text)
        if result['success']:
            print(f'Text: {text}')
            print(f'Predicted: {result["predicted_label"]} (confidence: {result["confidence"]:.2%})')
            print()
else:
    print('Classifier not trained yet. Run train/train_text_classifier.py first.')